# 🔍 Parquet File Inspector

Menampilkan informasi lengkap sebuah file Parquet:
metadata, schema, row groups, kompresi, statistik, dan preview data.

In [29]:
from collections import Counter
from datetime import datetime
from pathlib import Path
import struct
import io
import fastparquet as fp

import humanize
import pyarrow.parquet as pq
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

In [30]:
# ── Konfigurasi ───────────────────────────────────────────────────────────────

PARQUET_PATH = "./DataGPS_parquet/2022_04_April.parquet"   # ← ganti dengan path file Parquet kamu
N_PREVIEW    = 5                # jumlah baris untuk head & tail preview

---
## Helper functions

In [31]:
def make_table(title: str, columns: list[tuple]) -> Table:
    """Buat Rich Table dengan kolom dari list of (header, style, kwargs)."""
    t = Table(title=title, show_lines=True, header_style="bold magenta")
    for header, style, kwargs in columns:
        t.add_column(header, style=style, **kwargs)
    return t


def kv_table(rows: list[tuple]) -> Table:
    """Buat tabel key-value sederhana tanpa header."""
    t = Table(show_header=False, box=None, padding=(0, 2))
    t.add_column(style="bold yellow", no_wrap=True)
    t.add_column(style="white")
    for key, value in rows:
        t.add_row(key, value)
    return t


def ascii_bar(fraction: float, width: int = 30) -> str:
    """Bar ASCII proporsional. fraction antara 0.0–1.0."""
    return "\u2588" * max(0, int(fraction * width))


def decode_bytes(value) -> str:
    """Decode bytes ke str, atau kembalikan apa adanya jika sudah str."""
    return value.decode() if isinstance(value, bytes) else str(value)


def compression_ratio(compressed: int, uncompressed: int) -> float:
    return uncompressed / compressed if compressed else 0.0


def stat_value(stats, attr: str, fallback: str = "\u2014") -> str:
    """Ambil atribut dari column statistics; kembalikan fallback jika tidak ada."""
    if stats is None:
        return fallback
    val = getattr(stats, attr, None)
    return str(val) if val is not None else fallback

---
## Load file

In [32]:
path = Path(PARQUET_PATH)
assert path.exists(), f"File tidak ditemukan: {path.resolve()}"

pf        = pq.ParquetFile(path)
meta      = pf.metadata
schema    = pf.schema_arrow   # Arrow schema
pq_schema = pf.schema         # Parquet physical schema
file_stat = path.stat()

console.print(f"\u2705 File dimuat: [bold]{path.resolve()}[/bold]")

✅ File dimuat: /home/repal/Github/skripsi/DataGPS_parquet/2022_04_April.parquet

---
## 1 · Informasi File (OS Level)

In [33]:
def fmt_ts(ts: float) -> str:
    return datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M:%S")


console.print(kv_table([
    ("Path Absolut",  str(path.resolve())),
    ("Nama File",     path.name),
    ("Ekstensi",      path.suffix),
    ("Ukuran File",   f"{humanize.naturalsize(file_stat.st_size)} ({file_stat.st_size:,} bytes)"),
    ("Dibuat",        fmt_ts(file_stat.st_ctime)),
    ("Dimodifikasi",  fmt_ts(file_stat.st_mtime)),
]))

  Path Absolut    /home/repal/Github/skripsi/DataGPS_parquet/2022_04_April.parquet  
  Nama File       2022_04_April.parquet                                             
  Ekstensi        .parquet                                                          
  Ukuran File     9.7 MB (9,673,649 bytes)                                          
  Dibuat          2026-04-04 11:29:52                                               
  Dimodifikasi    2026-04-04 11:29:52                                               

---
## 2 · Parquet Metadata

In [34]:
console.print(kv_table([
    ("Dibuat Oleh",            meta.created_by or "(tidak diketahui)"),
    ("Format Version",         str(meta.format_version)),
    ("Total Baris",            f"{meta.num_rows:,}"),
    ("Total Kolom",            str(meta.num_columns)),
    ("Jumlah Row Groups",      str(meta.num_row_groups)),
    ("Serialized Footer Size", humanize.naturalsize(meta.serialized_size)),
]))

  Dibuat Oleh               DuckDB version v1.4.4 (build 6ddac802ff)  
  Format Version            1.0                                       
  Total Baris               1,048,571                                 
  Total Kolom               4                                         
  Jumlah Row Groups         9                                         
  Serialized Footer Size    4.7 kB                                    

---
## 3 · Schema Kolom

In [35]:
t = make_table("Arrow Schema", [
    ("#",              "dim",        {"width": 4}),
    ("Nama Kolom",     "bold white", {"no_wrap": True}),
    ("Arrow Type",     "green",      {}),
    ("Parquet Type",   "cyan",       {}),
    ("Nullable",       "yellow",     {"width": 8}),
    ("Metadata Kolom", "dim",        {}),
])

for i, field in enumerate(schema):
    col_meta = ""
    if field.metadata:
        col_meta = ", ".join(
            f"{decode_bytes(k)}={decode_bytes(v)}"
            for k, v in field.metadata.items()
        )[:80]

    t.add_row(
        str(i),
        field.name,
        str(field.type),
        str(pq_schema.column(i).physical_type),
        "\u2713" if field.nullable else "\u2717",
        col_meta,
    )

console.print(t)

                                Arrow Schema                                 
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ #    ┃ Nama Kolom ┃ Arrow Type ┃ Parquet Type ┃ Nullable ┃ Metadata Kolom ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ 0    │ maid       │ string     │ BYTE_ARRAY   │ ✓        │                │
├──────┼────────────┼────────────┼──────────────┼──────────┼────────────────┤
│ 1    │ latitude   │ double     │ DOUBLE       │ ✓        │                │
├──────┼────────────┼────────────┼──────────────┼──────────┼────────────────┤
│ 2    │ longitude  │ double     │ DOUBLE       │ ✓        │                │
├──────┼────────────┼────────────┼──────────────┼──────────┼────────────────┤
│ 3    │ timestamp  │ int64      │ INT64        │ ✓        │                │
└──────┴────────────┴────────────┴──────────────┴──────────┴────────────────┘

---
## 4 · Row Groups

In [36]:
def row_group_sizes(rg) -> tuple[int, int]:
    """Kembalikan (total_compressed, total_uncompressed) untuk satu row group."""
    compressed   = sum(rg.column(c).total_compressed_size   for c in range(rg.num_columns))
    uncompressed = sum(rg.column(c).total_uncompressed_size for c in range(rg.num_columns))
    return compressed, uncompressed


t = make_table(f"Row Groups ({meta.num_row_groups} total)", [
    ("RG #",         "dim",   {"width": 5}),
    ("Baris",        "white", {}),
    ("Ukuran Komp.", "green", {}),
    ("Rasio Komp.",  "cyan",  {}),
])

for i in range(meta.num_row_groups):
    rg           = meta.row_group(i)
    comp, uncomp = row_group_sizes(rg)
    t.add_row(
        str(i),
        f"{rg.num_rows:,}",
        humanize.naturalsize(comp),
        f"{compression_ratio(comp, uncomp):.2f}x",
    )

console.print(t)

              Row Groups (9 total)              
┏━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ RG #  ┃ Baris   ┃ Ukuran Komp. ┃ Rasio Komp. ┃
┡━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 0     │ 122,880 │ 1.1 MB       │ 2.86x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 1     │ 122,880 │ 1.1 MB       │ 2.85x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 2     │ 122,880 │ 1.1 MB       │ 2.87x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 3     │ 122,880 │ 1.1 MB       │ 2.87x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 4     │ 122,880 │ 1.0 MB       │ 2.26x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 5     │ 122,880 │ 1.1 MB       │ 2.86x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 6     │ 122,880 │ 1.1 MB       │ 2.86x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 7     │ 122,880 │ 1.1 MB       │ 2.85x       │
├───────┼─────────┼──────────────┼─────────────┤
│ 8     │ 65,531  │ 575.3 kB     │ 2.81x       │
└───────┴─────────┴──────────────┴─────────────┘

---
## 5 · Column Chunks (Row Group 0)

In [37]:
rg0 = meta.row_group(0)

t = make_table("Column Chunks \u2014 Row Group 0", [
    ("#",            "dim",     {"width": 4}),
    ("Kolom",        "white",   {"no_wrap": True}),
    ("Kompresi",     "green",   {}),
    ("Encoding",     "cyan",    {}),
    ("Compressed",   "yellow",  {}),
    ("Uncompressed", "yellow",  {}),
    ("Rasio",        "magenta", {}),
    ("Null Count",   "red",     {}),
])

for i in range(rg0.num_columns):
    cc        = rg0.column(i)
    comp      = cc.total_compressed_size
    uncomp    = cc.total_uncompressed_size
    encodings = ", ".join(str(e) for e in cc.encodings)[:30]

    t.add_row(
        str(i),
        cc.path_in_schema,
        str(cc.compression),
        encodings,
        humanize.naturalsize(comp),
        humanize.naturalsize(uncomp),
        f"{compression_ratio(comp, uncomp):.2f}x",
        stat_value(cc.statistics, "null_count"),
    )

console.print(t)

                                    Column Chunks — Row Group 0                                    
┏━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┓
┃ #    ┃ Kolom     ┃ Kompresi ┃ Encoding         ┃ Compressed ┃ Uncompressed ┃ Rasio ┃ Null Count ┃
┡━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━┩
│ 0    │ maid      │ ZSTD     │ PLAIN_DICTIONARY │ 324.6 kB   │ 715.1 kB     │ 2.20x │ 0          │
├──────┼───────────┼──────────┼──────────────────┼────────────┼──────────────┼───────┼────────────┤
│ 1    │ latitude  │ ZSTD     │ PLAIN            │ 254.7 kB   │ 983.1 kB     │ 3.86x │ 0          │
├──────┼───────────┼──────────┼──────────────────┼────────────┼──────────────┼───────┼────────────┤
│ 2    │ longitude │ ZSTD     │ PLAIN_DICTIONARY │ 216.5 kB   │ 360.7 kB     │ 1.67x │ 0          │
├──────┼───────────┼──────────┼──────────────────┼────────────┼──────────────┼───────┼────────────┤
│ 3    │ timestamp │ ZSTD     │ PLAIN            │ 267.1 kB   │ 983.1 kB     │ 3.68x │ 0          │
└──────┴───────────┴──────────┴──────────────────┴────────────┴──────────────┴───────┴────────────┘

---
## 6 · Min/Max Statistics per Kolom (RG 0)

In [38]:
t = make_table("Column Statistics \u2014 Row Group 0", [
    ("Kolom",      "white",  {"no_wrap": True}),
    ("Min",        "green",  {}),
    ("Max",        "green",  {}),
    ("Null Count", "red",    {}),
    ("Has MinMax", "yellow", {}),
])

for i in range(rg0.num_columns):
    cc    = rg0.column(i)
    stats = cc.statistics

    if stats and stats.has_min_max:
        mn, mx, has_mm = str(stats.min)[:40], str(stats.max)[:40], "\u2713"
    else:
        mn, mx, has_mm = "\u2014", "\u2014", "\u2717"

    t.add_row(
        cc.path_in_schema,
        mn, mx,
        stat_value(stats, "null_count"),
        has_mm,
    )

console.print(t)

                                          Column Statistics — Row Group 0                                          
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Kolom     ┃ Min                                 ┃ Max                                 ┃ Null Count ┃ Has MinMax ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ maid      │ 00001c73-1c73-e657-efda-af77b36cb4… │ 1dddd715-526b-4cd4-bcc5-29752c4723… │ 0          │ ✓          │
├───────────┼─────────────────────────────────────┼─────────────────────────────────────┼────────────┼────────────┤
│ latitude  │ -8.15925                            │ -7.58934                            │ 0          │ ✓          │
├───────────┼─────────────────────────────────────┼─────────────────────────────────────┼────────────┼────────────┤
│ longitude │ 110.03453063964844                  │ 110.81354                           │ 0          │ ✓          │
├───────────┼─────────────────────────────────────┼─────────────────────────────────────┼────────────┼────────────┤
│ timestamp │ 1648746007                          │ 1651337997                          │ 0          │ ✓          │
└───────────┴─────────────────────────────────────┴─────────────────────────────────────┴────────────┴────────────┘

---
## 7 · Distribusi Tipe Data

In [39]:
type_counts = Counter(str(field.type) for field in schema)
total_cols  = sum(type_counts.values())

t = make_table("Distribusi Tipe Data", [
    ("Tipe Data",   "green",  {}),
    ("Jml. Kolom", "white",  {}),
    ("Proporsi",   "cyan",   {}),
    ("Bar",         "yellow", {}),
])

for dtype, count in type_counts.most_common():
    pct = count / total_cols
    t.add_row(dtype, str(count), f"{pct:.1%}", ascii_bar(pct))

console.print(t)

                 Distribusi Tipe Data                  
┏━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Tipe Data ┃ Jml. Kolom ┃ Proporsi ┃ Bar             ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ double    │ 2          │ 50.0%    │ ███████████████ │
├───────────┼────────────┼──────────┼─────────────────┤
│ string    │ 1          │ 25.0%    │ ███████         │
├───────────┼────────────┼──────────┼─────────────────┤
│ int64     │ 1          │ 25.0%    │ ███████         │
└───────────┴────────────┴──────────┴─────────────────┘

---
## 8 · Ringkasan Kompresi

In [40]:
total_comp   = sum(
    meta.row_group(rg).column(c).total_compressed_size
    for rg in range(meta.num_row_groups)
    for c  in range(meta.row_group(rg).num_columns)
)
total_uncomp = sum(
    meta.row_group(rg).column(c).total_uncompressed_size
    for rg in range(meta.num_row_groups)
    for c  in range(meta.row_group(rg).num_columns)
)

ratio       = compression_ratio(total_comp, total_uncomp)
space_saved = 1 - (total_comp / total_uncomp) if total_uncomp else 0
avg_rows_rg = meta.num_rows / meta.num_row_groups if meta.num_row_groups else 0

console.print(kv_table([
    ("Ukuran Terkompresi",       humanize.naturalsize(total_comp)),
    ("Ukuran Tidak Terkompresi", humanize.naturalsize(total_uncomp)),
    ("Rasio Kompresi",           f"{ratio:.3f}x"),
    ("Penghematan Ruang",        f"{space_saved:.1%}"),
    ("Ukuran File di Disk",      humanize.naturalsize(file_stat.st_size)),
    ("Rata-rata Baris / RG",     f"{avg_rows_rg:,.0f}"),
]))

  Ukuran Terkompresi          9.1 MB   
  Ukuran Tidak Terkompresi    25.4 MB  
  Rasio Kompresi              2.788x   
  Penghematan Ruang           64.1%    
  Ukuran File di Disk         9.7 MB   
  Rata-rata Baris / RG        116,508  

---
## 9 · Preview Data

In [41]:
table_head = pf.read_row_group(0)
df_head     = table_head.slice(0, N_PREVIEW).to_pandas()

num_row_groups = pf.num_row_groups
table_tail = pf.read_row_group(num_row_groups - 1)
df_tail = table_tail.slice(max(0, table_tail.num_rows - N_PREVIEW)).to_pandas()

console.print(f"[bold]HEAD \u2014 {N_PREVIEW} baris pertama[/bold]")
console.print(df_head.to_string())
console.print()
console.print(f"[bold]TAIL \u2014 {N_PREVIEW} baris terakhir[/bold]")
console.print(df_tail.to_string())

HEAD — 5 baris pertama

maid  latitude   longitude   timestamp
0  00001c73-1c73-e657-efda-af77b36cb417  -7.89562  110.336830  1650361518
1  00004932-52b8-48d3-977c-bf30695945cb  -7.70400  110.406601  1651132492
2  00012bf3-7a4d-4bb1-a345-b3389ffd5688  -7.62990  110.423800  1648809892
3  00012bf3-7a4d-4bb1-a345-b3389ffd5688  -7.62990  110.423798  1648815642
4  00012bf3-7a4d-4bb1-a345-b3389ffd5688  -7.62990  110.423798  1648906676

TAIL — 5 baris terakhir

maid  latitude   longitude   timestamp
0  fffe5d7e-5d7e-235e-5c71-e88916b31504 -7.807503  110.342812  1649656725
1  fffe5d7e-5d7e-235e-5c71-e88916b31504 -7.807510  110.342812  1649656881
2  fffecf56-cf56-dc70-fc2e-ae609b3bf089 -7.781800  110.357498  1648860124
3  fffecf56-cf56-dc70-fc2e-ae609b3bf089 -7.781800  110.357498  1648860124
4  ffff5d3e-5d3e-7294-8a08-11eecee08959 -7.719400  110.357101  1649491319

---
## 10 · Ringkasan Akhir

In [42]:
summary = "\n".join([
    f"[bold]File      :[/bold] {path.name}",
    f"[bold]Ukuran    :[/bold] {humanize.naturalsize(file_stat.st_size)}",
    f"[bold]Baris     :[/bold] {meta.num_rows:,}",
    f"[bold]Kolom     :[/bold] {meta.num_columns}",
    f"[bold]Row Groups:[/bold] {meta.num_row_groups}",
    f"[bold]Rasio Komp:[/bold] {ratio:.3f}x  |  hemat {space_saved:.1%}",
    f"[bold]Creator   :[/bold] {meta.created_by or '\u2014'}",
])

console.print(Panel(summary, title="[bold green]\u2705 Inspeksi Selesai[/bold green]", expand=False))

╭──────────────── ✅ Inspeksi Selesai ─────────────────╮
│ File      : 2022_04_April.parquet                    │
│ Ukuran    : 9.7 MB                                   │
│ Baris     : 1,048,571                                │
│ Kolom     : 4                                        │
│ Row Groups: 9                                        │
│ Rasio Komp: 2.788x  |  hemat 64.1%                   │
│ Creator   : DuckDB version v1.4.4 (build 6ddac802ff) │
╰──────────────────────────────────────────────────────╯

# 11. Anatomi Binary File

In [43]:
# ── F1 · Anatomi Binary File ──────────────────────────────────────────────────
# Format Parquet: [PAR1][data...][footer][footer_len: 4 bytes][PAR1]

PARQUET_MAGIC = b"PAR1"

with open(path, "rb") as f:
    magic_start      = f.read(4)
    f.seek(-8, 2)                        # 8 bytes dari akhir
    footer_len_bytes = f.read(4)
    magic_end        = f.read(4)
    footer_len       = struct.unpack("<i", footer_len_bytes)[0]

file_size     = file_stat.st_size
footer_offset = file_size - 8 - footer_len
data_section  = footer_offset - 4       # dikurangi magic awal

console.print(kv_table([
    ("Magic Start",        f"{magic_start!r}  {'✓ valid' if magic_start == PARQUET_MAGIC else '✗ INVALID'}"),
    ("Magic End",          f"{magic_end!r}  {'✓ valid' if magic_end == PARQUET_MAGIC else '✗ INVALID'}"),
    ("File Size",          humanize.naturalsize(file_size)),
    ("Footer Length",      f"{footer_len:,} bytes  ({humanize.naturalsize(footer_len)})"),
    ("Footer Offset",      f"byte {footer_offset:,}"),
    ("Data Section Size",  humanize.naturalsize(data_section)),
    ("Footer / File %",    f"{footer_len / file_size:.3%}"),
    ("Data / File %",      f"{data_section / file_size:.3%}"),
]))

  Magic Start          b'PAR1'  ✓ valid       
  Magic End            b'PAR1'  ✓ valid       
  File Size            9.7 MB                 
  Footer Length        4,747 bytes  (4.7 kB)  
  Footer Offset        byte 9,668,894         
  Data Section Size    9.7 MB                 
  Footer / File %      0.049%                 
  Data / File %        99.951%                

# 12. Footer Raw Bytes

In [44]:
# ── F2 · Footer Raw Bytes ─────────────────────────────────────────────────────

DUMP_BYTES = 64   # berapa byte footer yang mau ditampilkan

with open(path, "rb") as f:
    f.seek(footer_offset)
    footer_raw = f.read(DUMP_BYTES)

console.print(f"[bold]Footer — {DUMP_BYTES} byte pertama (hex dump)[/bold]")
console.print()

# Tampilkan dalam format 16 byte per baris, hex + ASCII
for i in range(0, len(footer_raw), 16):
    chunk    = footer_raw[i : i + 16]
    hex_part = " ".join(f"{b:02x}" for b in chunk)
    asc_part = "".join(chr(b) if 32 <= b < 127 else "." for b in chunk)
    console.print(f"  [dim]{footer_offset + i:08x}[/dim]  [cyan]{hex_part:<47}[/cyan]  [yellow]{asc_part}[/yellow]")

Footer — 64 byte pertama (hex dump)

0093891e  15 02 19 5c 35 00 18 0d 64 75 63 6b 64 62 5f 73  ...\5...duckdb_s

0093892e  63 68 65 6d 61 15 08 00 15 0c 25 02 18 04 6d 61  chema.....%...ma

0093893e  69 64 25 00 00 15 0a 25 02 18 08 6c 61 74 69 74  id%....%...latit

0093894e  75 64 65 00 15 0a 25 02 18 09 6c 6f 6e 67 69 74  ude...%...longit

# 13. Column Chunk Byte Map

In [45]:
# ── F3 · Column Chunk Byte Map ────────────────────────────────────────────────

t = make_table("Column Chunk Byte Map (semua Row Groups)", [
    ("RG",           "dim",     {"width": 4}),
    ("Kolom",        "white",   {"no_wrap": True}),
    ("File Offset",  "cyan",    {}),
    ("Dict Page @",  "yellow",  {}),    # None = tidak pakai dictionary encoding
    ("Data Page @",  "green",   {}),
    ("Comp. Size",   "magenta", {}),
])

for rg_idx in range(meta.num_row_groups):
    rg = meta.row_group(rg_idx)
    for c in range(rg.num_columns):
        cc = rg.column(c)
        dict_offset = cc.dictionary_page_offset
        data_offset = cc.data_page_offset
        t.add_row(
            str(rg_idx),
            cc.path_in_schema,
            f"{cc.file_offset:,}",
            f"{dict_offset:,}" if dict_offset is not None else "[dim]—[/dim]",
            f"{data_offset:,}",
            humanize.naturalsize(cc.total_compressed_size),
        )

console.print(t)

                 Column Chunk Byte Map (semua Row Groups)                  
┏━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ RG   ┃ Kolom     ┃ File Offset ┃ Dict Page @ ┃ Data Page @ ┃ Comp. Size ┃
┡━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 0    │ maid      │ 0           │ 4           │ 279,269     │ 324.6 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 0    │ latitude  │ 0           │ —           │ 324,573     │ 254.7 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 0    │ longitude │ 0           │ 579,303     │ 638,056     │ 216.5 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 0    │ timestamp │ 0           │ —           │ 795,775     │ 267.1 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 1    │ maid      │ 0           │ 1,062,858   │ 1,350,845   │ 335.2 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 1    │ latitude  │ 0           │ —           │ 1,398,062   │ 253.5 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 1    │ longitude │ 0           │ 1,651,525   │ 1,714,559   │ 220.1 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 1    │ timestamp │ 0           │ —           │ 1,871,614   │ 271.0 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 2    │ maid      │ 0           │ 2,142,609   │ 2,417,549   │ 320.2 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 2    │ latitude  │ 0           │ —           │ 2,462,776   │ 250.2 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 2    │ longitude │ 0           │ 2,712,936   │ 2,770,844   │ 215.6 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 2    │ timestamp │ 0           │ —           │ 2,928,581   │ 267.4 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 3    │ maid      │ 0           │ 3,195,958   │ 3,486,777   │ 337.8 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 3    │ latitude  │ 0           │ —           │ 3,533,795   │ 247.0 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 3    │ longitude │ 0           │ 3,780,757   │ 3,839,100   │ 213.6 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 3    │ timestamp │ 0           │ —           │ 3,994,349   │ 271.6 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 4    │ maid      │ 0           │ 4,265,971   │ 4,505,439   │ 279.8 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 4    │ latitude  │ 0           │ 4,545,760   │ 4,681,274   │ 305.1 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 4    │ longitude │ 0           │ 4,850,880   │ 4,907,589   │ 199.8 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 4    │ timestamp │ 0           │ —           │ 5,050,638   │ 253.4 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 5    │ maid      │ 0           │ 5,304,010   │ 5,599,471   │ 343.3 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 5    │ latitude  │ 0           │ —           │ 5,647,267   │ 248.0 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 5    │ longitude │ 0           │ 5,895,243   │ 5,955,753   │ 215.7 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 5    │ timestamp │ 0           │ —           │ 6,110,990   │ 274.1 kB   │
├──────┼───────────┼─────────────┼─────────────┼─────────────┼────────────┤
│ 6    │ maid      │ 0           │ 6,385,088   │

# 14. Dictionary Encoding Detection

In [46]:
# ── F4 · Dictionary Encoding Detection ───────────────────────────────────────

t = make_table("Dictionary Encoding per Kolom (semua RG)", [
    ("Kolom",         "white",  {"no_wrap": True}),
    ("RG",            "dim",    {"width": 4}),
    ("Dict Page?",    "yellow", {}),
    ("Dict Offset",   "cyan",   {}),
    ("Encodings",     "green",  {}),
    ("Efek Kompresi", "magenta",{}),
])

for c in range(meta.num_columns):
    col_name = schema.field(c).name
    for rg_idx in range(meta.num_row_groups):
        cc          = meta.row_group(rg_idx).column(c)
        has_dict    = cc.dictionary_page_offset is not None
        dict_offset = str(cc.dictionary_page_offset) if has_dict else "—"
        encodings   = ", ".join(str(e) for e in cc.encodings)
        comp        = cc.total_compressed_size
        uncomp      = cc.total_uncompressed_size
        ratio_str   = f"{compression_ratio(comp, uncomp):.2f}x"

        t.add_row(
            col_name,
            str(rg_idx),
            "[green]✓ Ya[/green]" if has_dict else "[dim]✗ Tidak[/dim]",
            dict_offset,
            encodings,
            ratio_str,
        )

console.print(t)

                     Dictionary Encoding per Kolom (semua RG)                     
┏━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Kolom     ┃ RG   ┃ Dict Page? ┃ Dict Offset ┃ Encodings        ┃ Efek Kompresi ┃
┡━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ maid      │ 0    │ ✓ Ya       │ 4           │ PLAIN_DICTIONARY │ 2.20x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 1    │ ✓ Ya       │ 1062858     │ PLAIN_DICTIONARY │ 2.20x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 2    │ ✓ Ya       │ 2142609     │ PLAIN_DICTIONARY │ 2.20x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 3    │ ✓ Ya       │ 3195958     │ PLAIN_DICTIONARY │ 2.21x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 4    │ ✓ Ya       │ 4265971     │ PLAIN_DICTIONARY │ 2.19x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 5    │ ✓ Ya       │ 5304010     │ PLAIN_DICTIONARY │ 2.20x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 6    │ ✓ Ya       │ 6385088     │ PLAIN_DICTIONARY │ 2.19x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 7    │ ✓ Ya       │ 7445227     │ PLAIN_DICTIONARY │ 2.20x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ maid      │ 8    │ ✓ Ya       │ 8519808     │ PLAIN_DICTIONARY │ 2.15x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 0    │ ✗ Tidak    │ —           │ PLAIN            │ 3.86x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 1    │ ✗ Tidak    │ —           │ PLAIN            │ 3.88x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 2    │ ✗ Tidak    │ —           │ PLAIN            │ 3.93x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 3    │ ✗ Tidak    │ —           │ PLAIN            │ 3.98x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 4    │ ✓ Ya       │ 4545760     │ PLAIN_DICTIONARY │ 1.36x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 5    │ ✗ Tidak    │ —           │ PLAIN            │ 3.96x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 6    │ ✗ Tidak    │ —           │ PLAIN            │ 3.80x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 7    │ ✗ Tidak    │ —           │ PLAIN            │ 3.83x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ latitude  │ 8    │ ✗ Tidak    │ —           │ PLAIN            │ 3.68x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ longitude │ 0    │ ✓ Ya       │ 579303      │ PLAIN_DICTIONARY │ 1.67x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ longitude │ 1    │ ✓ Ya       │ 1651525     │ PLAIN_DICTIONARY │ 1.68x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ longitude │ 2    │ ✓ Ya       │ 2712936     │ PLAIN_DICTIONARY │ 1.67x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ longitude │ 3    │ ✓ Ya       │ 3780757     │ PLAIN_DICTIONARY │ 1.68x         │
├───────────┼──────┼────────────┼─────────────┼──────────────────┼───────────────┤
│ longitude │ 4 

# 15. Encoding Frequency

In [47]:
# ── F5 · Encoding Frequency ───────────────────────────────────────────────────

encoding_counter: Counter = Counter()

for rg_idx in range(meta.num_row_groups):
    rg = meta.row_group(rg_idx)
    for c in range(rg.num_columns):
        for enc in rg.column(c).encodings:
            encoding_counter[str(enc)] += 1

total_enc = sum(encoding_counter.values())

t = make_table("Encoding Frequency (semua RG × semua kolom)", [
    ("Encoding Type", "green",  {}),
    ("Count",         "white",  {}),
    ("Proporsi",      "cyan",   {}),
    ("Bar",           "yellow", {}),
])

for enc, cnt in encoding_counter.most_common():
    pct = cnt / total_enc
    t.add_row(enc, str(cnt), f"{pct:.1%}", ascii_bar(pct))

console.print(t)

       Encoding Frequency (semua RG × semua kolom)       
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Encoding Type    ┃ Count ┃ Proporsi ┃ Bar             ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ PLAIN_DICTIONARY │ 19    │ 52.8%    │ ███████████████ │
├──────────────────┼───────┼──────────┼─────────────────┤
│ PLAIN            │ 17    │ 47.2%    │ ██████████████  │
└──────────────────┴───────┴──────────┴─────────────────┘